# DWBDA Assignment endsem

**Student:** RITIK DAS  
**Registration number:** 202400130

In [ ]:
%pip -q install pyspark pandas scikit-learn matplotlib

## 1. Spark - Products sold in each category

Using the Kaggle Sample Sales Data to find total products sold for each product line.

In [ ]:
!wget -q https://www.kaggle.com/api/v1/datasets/download/kyanyoga/sample-sales-data -O sales_data.zip
!unzip -oq sales_data.zip -d sales_data
!find sales_data -type f

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
    .appName("Spark Practical") \
    .master("local[*]") \
    .getOrCreate()

# Display Spark version
print("Spark Version:", spark.version)

# Read sales CSV file
sales = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("sales_data/sales_data_sample.csv")

# Display first 5 rows
sales.show(5)

In [ ]:
# Create a temporary view
sales.createOrReplaceTempView("sales")

# Use SQL query to find products sold in each category
result = spark.sql("""
    SELECT 
        PRODUCTLINE,
        SUM(QUANTITYORDERED) AS products_sold
    FROM sales
    GROUP BY PRODUCTLINE
    ORDER BY PRODUCTLINE
""")

# Display the result
result.show()

## 2. Multivariate business analysis

Using the UCI Wine Quality dataset. The data has 11 measurable attributes. We standardize them, reduce with PCA and group with K-Means clustering.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Load wine quality dataset from UCI
wine = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
    sep=";"
)

# Display total number of records
print("Number of records:", len(wine))

# Check missing values
print("Missing values:", wine.isnull().sum().sum())

# Display first 5 rows
print()

print(wine.head())

In [ ]:
# Select all columns except quality
features = wine.columns.drop("quality")

# Display feature names
print("Features:", list(features))

# Standardize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(wine[features])

# Apply PCA to reduce data into 2 components
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_data)

# Display explained variance
print("Explained variance:", pca.explained_variance_ratio_)

# Apply K-Means clustering with 3 clusters
kmeans = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=42
)

# Add cluster labels to the dataset
wine["cluster"] = kmeans.fit_predict(scaled_data)

# Add PCA components to the dataset
wine["pc1"] = pca_result[:, 0]
wine["pc2"] = pca_result[:, 1]

# Calculate average values for each cluster
print()
print("Cluster averages:")

cluster_average = wine.groupby("cluster")[list(features)].mean().round(2)

print(cluster_average)

In [ ]:
# Plot the clusters
plt.figure(figsize=(8, 6))

# Display each cluster with different labels
for c in [0, 1, 2]:

    data = wine[wine["cluster"] == c]

    plt.scatter(
        data["pc1"],
        data["pc2"],
        label="Cluster " + str(c),
        s=20
    )

# Add labels and title
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine samples - PCA with K-Means clusters")

# Display legend
plt.legend()

# Show the graph
plt.show()

## 3. Previous semester exam results with PCA

Analyzing student marks across 5 subjects. Grades are calculated from the average marks. PCA is used to visualize the data in 2 dimensions.

In [ ]:
# load exam results
marks=pd.read_csv("previous_semester_exam_results.csv")
print(marks)

In [ ]:
# List of subjects
subjects = [
    "english",
    "mathematics",
    "physics",
    "chemistry",
    "computer_science"
]

# Calculate average marks of each student
marks["average"] = marks[subjects].mean(axis=1).round(2)


# Function to calculate grade
def get_grade(avg):

    if avg >= 90:
        return "S"

    elif avg >= 80:
        return "A"

    elif avg >= 70:
        return "B"

    elif avg >= 60:
        return "C"

    elif avg >= 50:
        return "D"

    elif avg >= 40:
        return "E"

    else:
        return "F"


# Assign grades to students
marks["grade"] = marks["average"].apply(get_grade)


# Count students in each grade
print("Students in each grade:")

for grade in ["S", "A", "B", "C", "D", "E", "F"]:

    count = len(marks[marks["grade"] == grade])

    print(grade, ":", count)


# Calculate average marks in each subject
print()
print("Subject averages:")

for subject in subjects:

    average = round(marks[subject].mean(), 2)

    print(subject, ":", average)

In [ ]:
# apply PCA on exam marks
scaler=StandardScaler()
scaled_marks=scaler.fit_transform(marks[subjects])

pca=PCA(n_components=2)
pca_result=pca.fit_transform(scaled_marks)

marks["pc1"]=pca_result[:,0]
marks["pc2"]=pca_result[:,1]

# plot students by grade
plt.figure(figsize=(8,6))
for g in ["S","A","B","C","D","E","F"]:
    data=marks[marks["grade"]==g]
    plt.scatter(data["pc1"],data["pc2"],label=g,s=50)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Exam results - PCA visualization")
plt.legend()
plt.show()

## Conclusion

- Spark SQL was used to find total products sold for each product line category.
- PCA and K-Means clustering were applied on the wine quality dataset to identify 3 groups.
- Student grades were calculated from average marks and the results were visualized using PCA.